# Data Loading (Initial)

This notebook sets up the initial data-loading process by loading the raw Kickstarter dataset directly from the team's shared Google Drive.The loaded dataset is used for validation and will be used for exploration, cleaning, preprocessing, and feature engineering in later notebooks. No cleaning, preprocessing, feature engineering, or train/test splitting is performed in this notebook.

In [1]:
# General imports
import sys
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
!python3.12 -m pip install --upgrade requests

In [4]:
# util
from utils.get_google_drive_data import load_csv_from_drive
from utils.google_drive_client import get_drive_client

### Load Raw Dataset from Google Drive

The raw CSV was downloaded from Kaggle and uploaded to Google Drive. The Google Drive copy was verified against the original download using SHA-256, and both files produced the same checksum. This confirms that the files are identical and that the encoding issue was not caused by the Google Drive transfer.

The source Kickstarter CSV contains some byte sequences that are not valid UTF-8. Attempting to load the file with the default UTF-8 decoding raises a `UnicodeDecodeError` at byte `0x92`. This issue is present in the original dataset.

The shared Google Drive loading utility uses UTF-8 decoding with `encoding_errors="replace"` to allow the raw dataset to be loaded without modifying the original file. Any resulting replacement characters can be reviewed and addressed during the data-cleaning step.

In [5]:
# authenticate
service = get_drive_client()

In [6]:
df = load_csv_from_drive(service)

In [7]:
df.head()

,project id,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
0,39409,WHILE THE TREES SLEEP,http://www.kickstarter.com/projects/emiliesaba...,Film & Video,Short Film,"Columbia, MO",successful,10500.0,11545.0,1.099524,66,"Fri, 19 Aug 2011 19:28:17 -0000",7,"$25,$50,$100,$250,$500,$1,000,$2,500",10,2,30.00
1,126581,Educational Online Trading Card Game,http://www.kickstarter.com/projects/972789543/...,Games,Board & Card Games,"Maplewood, NJ",failed,4000.0,20.0,0.005000,2,"Mon, 02 Aug 2010 03:59:00 -0000",5,"$1,$5,$10,$25,$50",6,0,47.18
2,138119,STRUM,http://www.kickstarter.com/projects/185476022/...,Film & Video,Animation,"Los Angeles, CA",live,20000.0,56.0,0.002800,3,"Fri, 08 Jun 2012 00:00:31 -0000",10,"$1,$10,$25,$40,$50,$100,$250,$1,000,$1,337,$9,001",1,0,28.00
3,237090,GETTING OVER - One son's search to finally kno...,http://www.kickstarter.com/projects/charnick/g...,Film & Video,Documentary,"Los Angeles, CA",successful,6000.0,6535.0,1.089167,100,"Sun, 08 Apr 2012 02:14:00 -0000",13,"$1,$10,$25,$30,$50,$75,$85,$100,$110,$250,$500...",4,0,32.22
4,246101,The Launch of FlyeGrlRoyalty &quot;The New Nam...,http://www.kickstarter.com/projects/flyegrlroy...,Fashion,Fashion,"Novi, MI",failed,3500.0,0.0,0.000000,0,"Wed, 01 Jun 2011 15:25:39 -0000",6,"$10,$25,$50,$100,$150,$250",2,0,30.00


## Initial Dataset Validation

Verify that the shared dataset loaded successfully and matches the expected structure of the original Kickstarter dataset.

In [8]:
# count rows and col to compare to https://www.kaggle.com/datasets/parienza/kickstarter/data
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 45,957
Columns: 17


In [9]:
# general check if dataset is empty or not
if df.empty:
    raise ValueError("Dataset is empty.")
print("Dataset is not empty. Good to continue")

Dataset is not empty. Good to continue


In [10]:
if len(df) != 45_957: # expected row count
    raise ValueError(
        f"Expected {EXPECTED_ROWS:,} rows, but found {len(df):,}."
    )

print("Row count verified.")

Row count verified.


In [11]:
EXPECTED_COLUMNS = [
    "project id",
    "name",
    "url",
    "category",
    "subcategory",
    "location",
    "status",
    "goal",
    "pledged",
    "funded percentage",
    "backers",
    "funded date",
    "levels",
    "reward levels",
    "updates",
    "comments",
    "duration",
]

In [12]:
if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "Dataset columns do not match the expected Kickstarter dataset columns."
    )

print("Dataset columns verified.")

Dataset columns verified.


In [13]:
print("Columns:")
print(df.columns.tolist())

Columns:
['project id', 'name', 'url', 'category', 'subcategory', 'location', 'status', 'goal', 'pledged', 'funded percentage', 'backers', 'funded date', 'levels', 'reward levels', 'updates', 'comments', 'duration']


## Dataset Structure

Inspect the raw column data types and non-null counts before any cleaning or preprocessing is performed.

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45957 entries, 0 to 45956
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   project id         45957 non-null  int64  
 1   name               45957 non-null  object 
 2   url                45957 non-null  object 
 3   category           45957 non-null  object 
 4   subcategory        45957 non-null  object 
 5   location           44635 non-null  object 
 6   status             45957 non-null  object 
 7   goal               45957 non-null  float64
 8   pledged            45945 non-null  float64
 9   funded percentage  45957 non-null  float64
 10  backers            45957 non-null  int64  
 11  funded date        45957 non-null  object 
 12  levels             45957 non-null  int64  
 13  reward levels      45898 non-null  object 
 14  updates            45957 non-null  int64  
 15  comments           45957 non-null  int64  
 16  duration           459

## Missing Values

The following check provides an initial view of missing values in the raw dataset.

Missing values are only identified here. Decisions about how to handle them will be made during data cleaning.

In [15]:
missing_values = df.isna().sum().to_frame(name="missing")

missing_values["missing %"] = (
    df.isna().mean() * 100
).round(2)

missing_values

,missing,missing %
project id,0,0.00
name,0,0.00
url,0,0.00
category,0,0.00
subcategory,0,0.00
location,1322,2.88
status,0,0.00
goal,0,0.00
pledged,12,0.03
funded percentage,0,0.00


## Status Distribution

The raw dataset contains multiple Kickstarter campaign statuses.

This notebook does not filter campaign statuses. The appropriate modeling population and target definition will be handled in later data-cleaning and modeling preparation steps.

In [16]:
df["status"].value_counts(dropna=False)

status
successful    22969
failed        18996
live           3929
canceled         59
suspended         4
Name: count, dtype: int64

## Initial Data Loading Complete

The raw Kickstarter dataset has been successfully loaded from the team's shared Google Drive and its basic structure has been validated.

At this stage:

- The dataset contains the expected number of rows and columns.
- The expected raw columns are present.
- Missing values have been identified but not modified.
- Campaign statuses have been inspected but not filtered.
- No cleaning has been performed.
- No features have been transformed or engineered.
- No train/test split has been performed.

Further exploration and data preparation are handled in subsequent notebooks.